# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [2]:
#Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [3]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [4]:
#your code here
# Remove rows with missing values
spaceship = spaceship.dropna()

# Keep only the deck letter from Cabin
spaceship["Cabin"] = spaceship["Cabin"].str[0]

# Drop identifiers that should not be used as features
spaceship = spaceship.drop(
    columns=["PassengerId", "Name"]
)

# Separate features and target
X = spaceship.drop(columns="Transported")
y = spaceship["Transported"]

# Identify categorical and numerical columns
categorical_columns = X.select_dtypes(
    include=["object", "bool"]
).columns

numerical_columns = X.select_dtypes(
    exclude=["object", "bool"]
).columns

print("Categorical columns:")
print(categorical_columns)

print("\nNumerical columns:")
print(numerical_columns)

Categorical columns:
Index(['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP'], dtype='str')

Numerical columns:
Index(['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck'], dtype='str')


C:\Users\stefa\AppData\Local\Temp\ipykernel_38108\2482664236.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(


**Perform Train Test Split**

In [7]:
#your code here
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=28,
    stratify=y
)

# One-hot encode categorical features
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

X_train_categorical = pd.DataFrame(
    encoder.fit_transform(X_train[categorical_columns]),
    columns=encoder.get_feature_names_out(categorical_columns),
    index=X_train.index
)

X_test_categorical = pd.DataFrame(
    encoder.transform(X_test[categorical_columns]),
    columns=encoder.get_feature_names_out(categorical_columns),
    index=X_test.index
)

# Scale numerical features
scaler = StandardScaler()

X_train_numerical = pd.DataFrame(
    scaler.fit_transform(X_train[numerical_columns]),
    columns=numerical_columns,
    index=X_train.index
)

X_test_numerical = pd.DataFrame(
    scaler.transform(X_test[numerical_columns]),
    columns=numerical_columns,
    index=X_test.index
)

# Combine numerical and encoded categorical features
X_train_processed = pd.concat(
    [X_train_numerical, X_train_categorical],
    axis=1
)

X_test_processed = pd.concat(
    [X_test_numerical, X_test_categorical],
    axis=1
)

print("Training shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)

Training shape: (5284, 23)
Test shape: (1322, 23)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [8]:
#your code here
model_results = []

# Bagging: samples can repeat
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=28),
    n_estimators=100,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

bagging.fit(X_train_processed, y_train)

bagging_predictions = bagging.predict(X_test_processed)

bagging_accuracy = accuracy_score(
    y_test,
    bagging_predictions
)

model_results.append({
    "Model": "Bagging",
    "Test accuracy": bagging_accuracy
})

# Pasting: samples cannot repeat
pasting = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=28),
    n_estimators=100,
    bootstrap=False,
    random_state=28,
    n_jobs=-1
)

pasting.fit(X_train_processed, y_train)

pasting_predictions = pasting.predict(X_test_processed)

pasting_accuracy = accuracy_score(
    y_test,
    pasting_predictions
)

model_results.append({
    "Model": "Pasting",
    "Test accuracy": pasting_accuracy
})

print("Bagging accuracy:", bagging_accuracy)
print("Pasting accuracy:", pasting_accuracy)

Bagging accuracy: 0.7708018154311649
Pasting accuracy: 0.7360060514372163


- Random Forests

In [10]:
#your code here
random_forest = RandomForestClassifier(
    n_estimators=200,
    random_state=28,
    n_jobs=-1
)

random_forest.fit(X_train_processed, y_train)

random_forest_predictions = random_forest.predict(
    X_test_processed
)

random_forest_accuracy = accuracy_score(
    y_test,
    random_forest_predictions
)

model_results.append({
    "Model": "Random Forest",
    "Test accuracy": random_forest_accuracy
})

print("Random Forest accuracy:", random_forest_accuracy)

Random Forest accuracy: 0.7753403933434191


- Gradient Boosting

In [11]:
#your code here
gradient_boosting = GradientBoostingClassifier(
    n_estimators=100,
    random_state=28
)

gradient_boosting.fit(X_train_processed, y_train)

gradient_boosting_predictions = gradient_boosting.predict(
    X_test_processed
)

gradient_boosting_accuracy = accuracy_score(
    y_test,
    gradient_boosting_predictions
)

model_results.append({
    "Model": "Gradient Boosting",
    "Test accuracy": gradient_boosting_accuracy
})

print("Gradient Boosting accuracy:", gradient_boosting_accuracy)

Gradient Boosting accuracy: 0.7745839636913767


- Adaptive Boosting

In [13]:
#your code here
adaboost = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=1,
        random_state=28
    ),
    n_estimators=100,
    random_state=28
)

adaboost.fit(X_train_processed, y_train)

adaboost_predictions = adaboost.predict(
    X_test_processed
)

adaboost_accuracy = accuracy_score(
    y_test,
    adaboost_predictions
)

model_results.append({
    "Model": "AdaBoost",
    "Test accuracy": adaboost_accuracy
})

print("AdaBoost accuracy:", adaboost_accuracy)

AdaBoost accuracy: 0.7692889561270801


Which model is the best and why?

In [14]:
#comment here
model_comparison = pd.DataFrame(model_results)

model_comparison = model_comparison.sort_values(
    by="Test accuracy",
    ascending=False
)

model_comparison = model_comparison.reset_index(drop=True)

display(
    model_comparison.style.format({
        "Test accuracy": "{:.2%}"
    })
)

best_model = model_comparison.iloc[0]

print(
    f"The best model is {best_model['Model']} "
    f"with a test accuracy of "
    f"{best_model['Test accuracy']:.2%}."
)

,Model,Test accuracy
0,Random Forest,78.21%
1,Random Forest,77.53%
2,Gradient Boosting,77.46%
3,Bagging,77.08%
4,AdaBoost,76.93%
5,AdaBoost,76.93%
6,Pasting,73.60%


The best model is Random Forest with a test accuracy of 78.21%.
